In [40]:
import os

from openai import OpenAI

openai_client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)


In [ ]:
instructions = "make sure to add emojis"
user_prompt = "my name is على علوكة"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": user_prompt},
]

response = openai_client.responses.create(
    model="qwen/qwen3.8-27b", input=messages, reasoning={"effort": "none"}
)
print(response.output_text)
messages.extend(response.output)

# response = None
# for event in stream:
#     if event.type == "response.output_text.delta":
#         print(event.delta, end="")
#     if event.type == "response.completed":
#         response = event.response


Hello **Ali Alouka**! 👋 It’s nice to meet you.

How can I help you today? 😊


In [42]:
messages.append({"role": "user", "content": "what's my name?"})

response = openai_client.responses.create(
    model="qwen/qwen3.8-27b", input=messages, reasoning={"effort": "none"}
)


print(response.output_text)
messages.extend(response.output)
messages

Your name is **Ali Alouka** (على علوكة)! 🌟😊


[{'role': 'system', 'content': 'make sure to add emojis'},
 {'role': 'user', 'content': 'my name is على علوكة'},
 ResponseReasoningItem(id='resp_01m5fn1mxcec4vac37j6zh9pe4', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'),
 ResponseOutputMessage(id='msg_01m5fn1mxcec5a5wwtnhqejawk', content=[ResponseOutputText(annotations=[], text='Hello **Ali Alouka**! 👋 It’s nice to meet you.\n\nHow can I help you today? 😊', type='output_text', logprobs=None)], role='assistant', status='completed', type='message', phase=None),
 {'role': 'user', 'content': "what's my name?"},
 ResponseReasoningItem(id='resp_01m5fn1n5necfaje5106q08vqc', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'),
 ResponseOutputMessage(id='msg_01m5fn1n5necfs03ph7gkg4bty', content=[ResponseOutputText(annotations=[], text='Your name is **Ali Alouka** (على علوكة)! 🌟😊', type='output_text', logprobs=None)], role='assistant', status='completed', type='message

In [43]:
messages.append({"role": "user", "content": "what's 2 * 2 + 2? in one word"})

response = openai_client.responses.create(
    model="qwen/qwen3.8-27b", input=messages, reasoning={"effort": "none"}
)


print(response.output_text)
messages.extend(response.output)
messages

6️⃣


[{'role': 'system', 'content': 'make sure to add emojis'},
 {'role': 'user', 'content': 'my name is على علوكة'},
 ResponseReasoningItem(id='resp_01m5fn1mxcec4vac37j6zh9pe4', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'),
 ResponseOutputMessage(id='msg_01m5fn1mxcec5a5wwtnhqejawk', content=[ResponseOutputText(annotations=[], text='Hello **Ali Alouka**! 👋 It’s nice to meet you.\n\nHow can I help you today? 😊', type='output_text', logprobs=None)], role='assistant', status='completed', type='message', phase=None),
 {'role': 'user', 'content': "what's my name?"},
 ResponseReasoningItem(id='resp_01m5fn1n5necfaje5106q08vqc', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'),
 ResponseOutputMessage(id='msg_01m5fn1n5necfs03ph7gkg4bty', content=[ResponseOutputText(annotations=[], text='Your name is **Ali Alouka** (على علوكة)! 🌟😊', type='output_text', logprobs=None)], role='assistant', status='completed', type='message

# Self Written Drill
- call the openAI and do a full round trip convo

In [4]:
import os

import openai


def create_client(base_url="https://api.groq.com/openai/v1", api_key=None):
    return openai.OpenAI(
        base_url=base_url, api_key=api_key or os.environ["GROQ_API_KEY"]
    )


def chat(
    history,
    user_query,
    model="qwen/qwen3.8-27b",
    reasoning=None,
    max_output_tokens=300,
):
    history.append({"role": "user", "content": user_query})

    response = create_client().responses.create(
        model=model,
        input=history,
        reasoning=reasoning or {"effort": "none"},
        max_output_tokens=max_output_tokens,
    )
    history.extend(response.output)

    print(response.output_text)


system_instruction = "use as less tokens as possible and be very concise"
history = [{"role": "system", "content": system_instruction}]

chat(history, "15*5+5")


80


# OOP Refactor 

In [19]:
import os

import openai


class ClientManager:
    def __init__(self, base_url: str, api_key_env_var: str) -> None:
        self.base_url = base_url
        self.api_key = os.environ.get(api_key_env_var)
        if not self.api_key:
            raise ValueError(f"Environment variable '{api_key_env_var}' not found.")

    def create_client(self) -> openai.OpenAI:
        return openai.OpenAI(base_url=self.base_url, api_key=self.api_key)


class ChatSession:
    def __init__(
        self,
        client,
        model: str,
        system_instruction="Provide the final raw answer text only. Absolutely no explanations, no math steps, no formatting symbols, and no chat filler.",
        reasoning: str = "none",
        max_output_tokens: int = 30,
    ) -> None:
        self.client = client
        self.model = model
        self.reasoning = {"effort": reasoning}
        self.max_output_tokens = max_output_tokens
        self.history = [{"role": "system", "content": system_instruction}]

    def chat(self, user_query: str, model=None, reasoning=None, max_output_tokens=None):

        request = {"role": "user", "content": user_query}
        temp_history = list(self.history)
        temp_history.append(request)

        active_model = model or self.model
        active_reasoning = (
            {"effort": reasoning} if reasoning is not None else self.reasoning
        )
        active_token_limit = max_output_tokens or self.max_output_tokens

        response = self.client.responses.create(
            model=active_model,
            input=temp_history,
            reasoning=active_reasoning,
            max_output_tokens=active_token_limit,
        )
        self.history.append(request)
        self.history.extend(response.output)
        print(response.output_text)

        return response


In [2]:
# create client
client = ClientManager(
    base_url="https://api.groq.com/openai/v1", api_key_env_var="GROQ_API_KEY"
).create_client()

In [20]:
# start session and prompt
session = ChatSession(client, "qwen/qwen3.8-27b")


In [21]:
# test out memory on multi turns
session.chat("5*5")
session.chat("15*5+5")
session.chat("say hi")
session.chat("what was the first number u calculated?")
session.chat(
    "what was the second number u calculated?",
    reasoning="low",
    max_output_tokens=667,
).status

25
80
Hi
25
80


'completed'